# Empirical Evaluation of Incremental LOF for Anomaly Detection in Data Streams
*Deep and Reinforcement Learning – M.IA003, FEUP/FCUP*

---

**Algorithm under study:** Incremental Local Outlier Factor (LOF)  
**Baseline:** Half-Space Trees (HST)  
**Evaluation framework:** Prequential evaluation on synthetic and real-world streams

---

## Table of Contents

1. [Objective](#1-objective)
2. [Problem Context](#2-problem-context)
3. [Algorithms](#3-algorithms)
4. [Research Questions & Hypotheses](#4-research-questions--hypotheses)
5. [Experimental Design & Metrics](#5-experimental-design--metrics)
6. [Setup & Imports](#6-setup--imports)
7. [Datasets](#7-datasets)
8. [Experiments](#8-experiments)
   - [Exp 1 – Baseline comparison on the synthetic stream (H1, H2, H3)](#exp-1--baseline-comparison-on-the-synthetic-stream)
   - [Exp 2A – Effect of k under concept drift (H4)](#exp-2a--effect-of-k-under-concept-drift)
   - [Exp 2B – Effect of k in a mixed-density stationary stream (H4)](#exp-2b--effect-of-k-in-a-mixed-density-stationary-stream)
   - [Exp 3 – LOF in its ideal scenario: local anomalies (H1)](#exp-3--lof-in-its-ideal-scenario-local-anomalies)
   - [Exp 4 – Real-world validation: Credit Card Fraud (TODO)](#exp-4--real-world-validation-credit-card-fraud)
   - [Exp 5 – Runtime & Memory (TODO)](#exp-5--runtime--memory)
9. [Discussion & Conclusions](#9-discussion--conclusions)


---
## 1. Objective

The goal of this study is to evaluate the effectiveness of **Incremental LOF** for anomaly detection in data streams, and to compare its performance against a stream-native baseline (**Half-Space Trees**).

The study focuses on answering:
- Under which conditions does Incremental LOF perform well?
- What are its strengths and limitations in dynamic streaming environments?


---
## 2. Problem Context

Anomaly detection in data streams consists of identifying unusual or rare observations in a continuous flow of data. Unlike traditional settings, data arrives sequentially and cannot be stored entirely.

Key challenges:
- No clear train/test split — models are evaluated as data flows.
- Models must **learn incrementally** from a single pass over the data.
- The underlying data distribution may change over time (**concept drift**).

Algorithms must be efficient, adaptive, and capable of maintaining performance over time.


---
## 3. Algorithms

### 3.1 Incremental LOF (Algorithm Under Study)

**Core idea:** A point is considered anomalous if its local density is significantly lower than that of its neighbors. This allows detection of *contextual anomalies* — points that are unusual *relative to their local neighborhood*, but not necessarily globally extreme.

**Streaming adaptation:** For each point, the algorithm stores its k-nearest neighbors, reverse nearest neighbors, local reachability density (LRD), and LOF score. When a new point arrives, only the affected neighborhood is recomputed, avoiding a full global pass.

**Key assumptions:**
- Local density is meaningful and computable via a good distance metric.
- The choice of k defines the notion of "local" — small k is noisy, large k may miss fine-grained anomalies.
- **No built-in drift handling:** past data continues to influence density estimates even after distribution changes.

---

### 3.2 Half-Space Trees (Baseline)

**Core idea:** Recursively partitions the feature space into regions using random axis-aligned splits. Points in low-mass regions are considered anomalous.

**Why this baseline?** HST represents a fundamentally different approach:
- It is *stream-native*, designed for online processing.
- It does **not** rely on distances or neighborhood structures.
- It provides a strong contrast to LOF's density-based perspective.

| Aspect | Incremental LOF | Half-Space Trees |
|---|---|---|
| Core idea | Local density | Space partitioning |
| Uses distance | Yes | No |
| Perspective | Local | Global-ish |
| Drift handling | Weak (no forgetting) | Better (structure-based) |
| Complexity | Potentially high | More stable |


---
## 4. Research Questions & Hypotheses

**Central question:**
> Under which conditions is Incremental LOF an effective anomaly detector in data streams compared to Half-Space Trees?

### Hypotheses

| ID | Statement |
|---|---|
| **H1** | LOF performs better than HST for *local* anomalies (defined relative to their neighborhood). |
| **H2** | LOF performance degrades under *concept drift* due to lack of forgetting. |
| **H3** | HST is more *stable* in evolving streams. |
| **H4** | LOF is *sensitive to the choice of k* — both in overall performance and in how gracefully it degrades. |


---
## 5. Experimental Design & Metrics

### Evaluation Protocol: Prequential Evaluation

In data stream settings, traditional train/test splits do not apply. We use **prequential evaluation** (test-then-train):
1. The model predicts an anomaly score for instance `(x_i, y_i)`.
2. The prediction is evaluated against the true label.
3. The model is updated with the new instance.

To capture performance under evolving conditions we also use a **sliding window ROC AUC**, where only the most recent `w` observations contribute to the score.

### Metrics

| Metric | Rationale |
|---|---|
| **ROC AUC** | Ranking metric; suited for imbalanced datasets where accuracy is misleading. |
| **Windowed ROC AUC** | Reveals short-term adaptation behaviour and response to drift. |
| **Runtime / Memory** *(TODO)* | Required to assess streaming feasibility. |

> **Note:** Average Precision could complement ROC AUC (especially at extreme imbalance ratios), but is left for future work.


---
## 6. Setup & Imports


In [ ]:
from river import metrics, anomaly
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from collections import deque
from sklearn.metrics import roc_auc_score

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)


---
## 7. Datasets

Two complementary datasets are used:
- A **synthetic dataset** for controlled hypothesis testing.
- A **real-world dataset** (Credit Card Fraud) for external validity.


### 7.1 Synthetic Dataset

The stream is divided into four phases to directly probe the algorithm's assumptions:

| Phase | Behaviour | Tests |
|---|---|---|
| Stable | Fixed dense clusters, 5 % anomaly rate | LOF ideal conditions |
| Abrupt drift | Cluster centres shift suddenly | Adaptation to sudden change |
| Gradual drift | Clusters slowly move | Accumulation of stale density info |
| High anomaly rate | 15 % anomaly rate | Robustness under contamination |


In [ ]:
def generate_synthetic_stream(n_samples=4000, seed=42):
    np.random.seed(seed)
    random.seed(seed)
    stream = []

    # Phase 1 – Stable
    for _ in range(n_samples // 4):
        if random.random() < 0.05:
            x = np.random.uniform(-6, 6, size=2); y = 1
        else:
            center = random.choice([(0, 0), (3, 3)])
            x = np.random.normal(loc=center, scale=0.5, size=2); y = 0
        stream.append(({"x1": x[0], "x2": x[1]}, y, "stable"))

    # Phase 2 – Abrupt drift
    for _ in range(n_samples // 4):
        if random.random() < 0.05:
            x = np.random.uniform(-6, 6, size=2); y = 1
        else:
            center = random.choice([(6, 0), (0, 6)])
            x = np.random.normal(loc=center, scale=0.5, size=2); y = 0
        stream.append(({"x1": x[0], "x2": x[1]}, y, "abrupt"))

    # Phase 3 – Gradual drift
    for i in range(n_samples // 4):
        drift = i / (n_samples // 4)
        center = (3 * drift, 3 * drift)
        if random.random() < 0.05:
            x = np.random.uniform(-6, 6, size=2); y = 1
        else:
            x = np.random.normal(loc=center, scale=0.7, size=2); y = 0
        stream.append(({"x1": x[0], "x2": x[1]}, y, "gradual"))

    # Phase 4 – High anomaly rate
    for _ in range(n_samples // 4):
        if random.random() < 0.15:
            x = np.random.uniform(-6, 6, size=2); y = 1
        else:
            center = random.choice([(2, 2), (5, 5)])
            x = np.random.normal(loc=center, scale=0.6, size=2); y = 0
        stream.append(({"x1": x[0], "x2": x[1]}, y, "anomaly"))

    return stream


def stream_to_df(stream):
    return pd.DataFrame([
        {"x1": x["x1"], "x2": x["x2"], "label": y, "phase": phase}
        for x, y, phase in stream
    ])


synthetic_stream = generate_synthetic_stream()
df_synth = stream_to_df(synthetic_stream)
print(f"Stream size: {len(synthetic_stream)} | Anomaly rate: {df_synth['label'].mean():.2%}")


#### Visualisation: Per-Phase Structure

In [ ]:
PHASE_COLORS = {"stable": "steelblue", "abrupt": "orange",
               "gradual": "green", "anomaly": "red"}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, phase in zip(axes, ["stable", "abrupt", "gradual", "anomaly"]):
    sub = df_synth[df_synth["phase"] == phase]
    ax.scatter(sub[sub["label"]==0]["x1"], sub[sub["label"]==0]["x2"],
               s=8, alpha=0.4, label="Normal")
    ax.scatter(sub[sub["label"]==1]["x1"], sub[sub["label"]==1]["x2"],
               s=20, alpha=0.9, c="red", label="Anomaly")
    ax.set_title(f"Phase: {phase}")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.legend(fontsize=7)

plt.suptitle("Synthetic stream – per-phase structure", y=1.02)
plt.tight_layout()
plt.show()


**Observations:**
- **Stable:** Two tight clusters, anomalies are clearly sparse globally.
- **Abrupt:** Clusters jump to new locations — previously learned density structures become outdated.
- **Gradual:** A single cluster drifts continuously, creating transitional overlap.
- **High anomaly:** Denser anomaly contamination tests robustness under class imbalance.


#### Visualisation: Feature Evolution Over Time

In [ ]:
df_synth["t"] = range(len(df_synth))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, feat in zip(axes, ["x1", "x2"]):
    ax.scatter(df_synth["t"], df_synth[feat], s=2, alpha=0.3)
    for xv, lbl in zip([1000, 2000, 3000], ["Abrupt", "Gradual", "High anomaly"]):
        ax.axvline(xv, linestyle="--", color="gray")
        ax.text(xv + 30, ax.get_ylim()[1] * 0.9, lbl, fontsize=8, color="gray")
    ax.set_title(f"Feature {feat} over time")
    ax.set_xlabel("Instance index")

plt.suptitle("Feature evolution – drift phases are clearly visible")
plt.tight_layout()
plt.show()


### 7.2 Real Dataset – Credit Card Fraud

The [Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) is a standard benchmark processed as a sequential stream.

It is suitable because:
- It is **highly imbalanced** (fraud ≈ 0.17 %), requiring ranking metrics.
- It contains real-world noise and complexity not present in synthetic data.
- It can be streamed to simulate realistic online deployment.

> The `Time` column is dropped (it represents seconds since the first transaction and does not encode temporal patterns relevant to fraud). `Amount` is kept as a feature.


In [ ]:
def load_creditcard_stream(path):
    df = pd.read_csv(path)
    if "Time" in df.columns:
        df = df.drop(columns=["Time"])
    stream = []
    for _, row in df.iterrows():
        x = row.drop("Class").to_dict()
        y = int(row["Class"])
        stream.append((x, y))
    return stream

creditcard_stream = load_creditcard_stream("datasets/creditcard.csv")
labels_cc = pd.Series([y for _, y in creditcard_stream])
print(f"Total instances : {len(creditcard_stream):,}")
print(f"Fraud rate      : {labels_cc.mean():.4%}")


#### Credit Card – EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Class imbalance
counts = labels_cc.value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(["Normal", "Fraud"], counts.values)
axes[0].set_yscale("log")
axes[0].set_title("Class distribution (log scale)")
axes[0].set_ylabel("Count")

# 2. Transaction amount over time
amounts = [x["Amount"] for x, _ in creditcard_stream]
axes[1].scatter(range(len(amounts)), amounts, s=1, alpha=0.2)
axes[1].set_title("Amount over time")
axes[1].set_xlabel("Transaction index")
axes[1].set_ylabel("Amount")

# 3. Amount distribution
pd.Series(amounts).hist(bins=50, ax=axes[2])
axes[2].set_title("Amount distribution (heavy-tailed)")
axes[2].set_xlabel("Amount")

plt.suptitle("Credit Card dataset – exploratory analysis")
plt.tight_layout()
plt.show()


#### Credit Card – PCA Projection

In [ ]:
df_cc = pd.DataFrame([{**x, "Class": y} for x, y in creditcard_stream])
sample = df_cc.sample(5000, random_state=42)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(sample.drop(columns=["Class"]))
y_sample = sample["Class"].values

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[y_sample==0, 0], X_pca[y_sample==0, 1],
            s=3, alpha=0.3, label="Normal")
plt.scatter(X_pca[y_sample==1, 0], X_pca[y_sample==1, 1],
            s=12, alpha=0.7, c="red", label="Fraud")
plt.title("PCA projection (2D) – 5 000 sample points")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Explained variance by PC1+PC2: {pca.explained_variance_ratio_.sum():.2%}")


**Implications for the algorithms:**

The PCA projection reveals substantial overlap between normal and fraudulent transactions — anomalies are *not* clearly separable geometrically. Combined with the heavy-tailed amount distribution, this means:
- **LOF may struggle**: density differences are weak, and extreme amounts can cause false positives.
- **HST is expected to be more robust**: less dependent on metric structure.

This dataset is therefore a stress test rather than a showcase for either algorithm.


---
## 8. Experiments

### Shared Utilities


In [ ]:
# ── Prequential evaluation (cumulative ROC AUC) ──────────────────────────────

def run_experiment(stream, model):
    """
    Predict → Evaluate → Update for each instance.
    Returns cumulative ROC AUC over time.
    """
    roc_auc = metrics.ROCAUC()
    scores = []
    for item in stream:
        x, y = item[0], item[1]
        score = model.score_one(x)
        roc_auc.update(y, score)
        scores.append(roc_auc.get())
        model.learn_one(x)
    return scores


# ── Windowed ROC AUC ──────────────────────────────────────────────────────────

def run_experiment_with_window(stream, model, window_size=300):
    """
    Same predict→evaluate→update loop, but also tracks a sliding-window ROC AUC
    to reveal short-term adaptation behaviour.
    Returns (cumulative_scores, windowed_scores).
    """
    roc_auc = metrics.ROCAUC()
    cumulative_scores, windowed_scores = [], []
    y_window, score_window = deque(maxlen=window_size), deque(maxlen=window_size)

    for item in stream:
        x, y = item[0], item[1]
        score = model.score_one(x)

        roc_auc.update(y, score)
        cumulative_scores.append(roc_auc.get())

        y_window.append(y)
        score_window.append(score)

        if len(y_window) == window_size and len(set(y_window)) > 1:
            windowed_scores.append(roc_auc_score(list(y_window), list(score_window)))
        else:
            windowed_scores.append(np.nan)

        model.learn_one(x)

    return cumulative_scores, windowed_scores


# ── Phase boundary helpers ────────────────────────────────────────────────────

def add_phase_lines(ax, n, labels=("Stable", "Abrupt", "Gradual", "High\nAnomaly")):
    p1, p2, p3 = n // 4, n // 2, 3 * n // 4
    for xv in [p1, p2, p3]:
        ax.axvline(xv, linestyle="--", color="gray", linewidth=0.8)
    y_max = ax.get_ylim()[1]
    for xc, lbl in zip([p1/2, (p1+p2)/2, (p2+p3)/2, (p3+n)/2], labels):
        ax.text(xc, y_max * 0.97, lbl, ha="center", fontsize=8, color="dimgray")


---
### Exp 1 – Baseline Comparison on the Synthetic Stream

**Hypotheses tested:** H1, H2, H3  
**Goal:** Establish whether LOF and HST differ in overall performance and in how they respond to each phase of the stream.


In [ ]:
lof_exp1 = anomaly.LocalOutlierFactor(n_neighbors=10)
hst_exp1 = anomaly.HalfSpaceTrees(n_trees=10, height=8, window_size=250)

lof_scores_exp1 = run_experiment(synthetic_stream, lof_exp1)
hst_scores_exp1 = run_experiment(synthetic_stream, hst_exp1)

n = len(synthetic_stream)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(lof_scores_exp1, label="LOF (k=10)", linewidth=1.2)
ax.plot(hst_scores_exp1, label="HST", linewidth=1.2)
add_phase_lines(ax, n)
ax.set_title("Exp 1 – Cumulative ROC AUC over time (Synthetic stream)", pad=15)
ax.set_xlabel("Instance")
ax.set_ylabel("ROC AUC")
ax.legend()
plt.tight_layout()
plt.show()

# Summary table
summary_exp1 = pd.DataFrame({
    "Model": ["LOF (k=10)", "HST"],
    "Stable (mean)": [
        np.mean(lof_scores_exp1[:n//4]),
        np.mean(hst_scores_exp1[:n//4])
    ],
    "Abrupt (mean)": [
        np.mean(lof_scores_exp1[n//4:n//2]),
        np.mean(hst_scores_exp1[n//4:n//2])
    ],
    "Gradual (mean)": [
        np.mean(lof_scores_exp1[n//2:3*n//4]),
        np.mean(hst_scores_exp1[n//2:3*n//4])
    ],
    "High anomaly (mean)": [
        np.mean(lof_scores_exp1[3*n//4:]),
        np.mean(hst_scores_exp1[3*n//4:])
    ],
    "Final AUC": [lof_scores_exp1[-1], hst_scores_exp1[-1]]
})
print(summary_exp1.to_string(index=False))


**Interpretation:**
- **LOF** performs reasonably in the stable phase but degrades sharply at abrupt drift and never recovers. This is consistent with **H2** (LOF struggles under drift).
- **HST** remains stable and improves over time, consistent with **H3** (HST is more stable in evolving streams).
- The contrast confirms that drift, rather than anomaly difficulty, is the dominant factor limiting LOF here.

> ⚠️ *Note:* With k = 10, LOF performance in the stable phase is already moderate. Experiment 2B will show that a better k choice significantly improves the stable-phase baseline.


---
### Exp 2A – Effect of k Under Concept Drift

**Hypothesis tested:** H4  
**Goal:** Understand whether k-tuning can compensate for LOF's sensitivity to drift, or whether drift is the dominant failure mode regardless of k.


In [ ]:
k_values_2a = [5, 10, 20, 40]
k_results_2a = {}

for k in k_values_2a:
    lof_k = anomaly.LocalOutlierFactor(n_neighbors=k)
    k_results_2a[k] = run_experiment(synthetic_stream, lof_k)

n = len(synthetic_stream)
fig, ax = plt.subplots(figsize=(11, 4))
for k, scores in k_results_2a.items():
    ax.plot(scores, label=f"k={k}", linewidth=1.2)
add_phase_lines(ax, n)
ax.set_title("Exp 2A – Effect of k on LOF (drift stream)", pad=15)
ax.set_xlabel("Instance")
ax.set_ylabel("Cumulative ROC AUC")
ax.legend()
plt.tight_layout()
plt.show()

# Phase-wise summary
p1, p2, p3 = n // 4, n // 2, 3 * n // 4
rows = []
for k, scores in k_results_2a.items():
    rows.append({
        "k": k,
        "Stable": np.mean(scores[:p1]),
        "Abrupt": np.mean(scores[p1:p2]),
        "Gradual": np.mean(scores[p2:p3]),
        "High anomaly": np.mean(scores[p3:]),
        "Final AUC": scores[-1]
    })
print(pd.DataFrame(rows).to_string(index=False))


**Interpretation:**
- Larger k (40) consistently outperforms smaller k in the stable phase.
- After drift, **all k configurations converge to similarly low performance**, indicating that k-tuning cannot rescue LOF from distribution shift.
- The dominant limitation here is the absence of a forgetting mechanism, not parameter sensitivity.

> This motivates Experiment 2B: to properly isolate the role of k, we need a drift-free setting where local density structure matters.


---
### Exp 2B – Effect of k in a Mixed-Density Stationary Stream

**Hypothesis tested:** H4  
**Goal:** Isolate the effect of k by removing concept drift, designing a stream where anomaly detection depends purely on local density structure.

**Dataset design:**
- Dense cluster at origin (scale 0.20)
- Sparse cluster at (4, 0) (scale 0.75)
- Anomalies placed on a thin ring around the dense cluster (radius ≈ 0.75)

The ring anomalies are *locally* anomalous (low-density relative to the dense cluster), but they overlap the sparse cluster's density range. This forces LOF to find the right neighborhood granularity.


In [ ]:
def generate_k_sensitive_stream(n_samples=4000, seed=42):
    np.random.seed(seed)
    random.seed(seed)
    stream = []
    for _ in range(n_samples):
        r = random.random()
        if r < 0.55:                              # Dense normal cluster
            x = np.random.normal(loc=(0.0, 0.0), scale=0.20, size=2); y = 0; phase = "dense"
        elif r < 0.85:                            # Sparse normal cluster
            x = np.random.normal(loc=(4.0, 0.0), scale=0.75, size=2); y = 0; phase = "sparse"
        else:                                     # Ring anomalies
            angle = np.random.uniform(0, 2 * np.pi)
            radius = np.random.normal(loc=0.75, scale=0.08)
            x = np.array([radius * np.cos(angle), radius * np.sin(angle)]); y = 1; phase = "anomaly"
        stream.append(({"x1": float(x[0]), "x2": float(x[1])}, y, phase))
    return stream


stream_2b = generate_k_sensitive_stream()
df_2b = stream_to_df(stream_2b)

# Visualise
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_2b[df_2b["label"]==0]["x1"], df_2b[df_2b["label"]==0]["x2"],
           s=8, alpha=0.3, label="Normal")
ax.scatter(df_2b[df_2b["label"]==1]["x1"], df_2b[df_2b["label"]==1]["x2"],
           s=15, alpha=0.8, c="red", label="Anomaly")
ax.set_title("Exp 2B – Mixed-density stream")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
k_values_2b = [3, 5, 10, 20, 40, 80]
k_results_2b = {}

for k in k_values_2b:
    lof_k = anomaly.LocalOutlierFactor(n_neighbors=k)
    cum, win = run_experiment_with_window(stream_2b, lof_k, window_size=300)
    k_results_2b[k] = {"cumulative": cum, "windowed": win}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for k in k_values_2b:
    axes[0].plot(k_results_2b[k]["windowed"], label=f"k={k}", linewidth=1)
    axes[1].plot(k_results_2b[k]["cumulative"], label=f"k={k}", linewidth=1)

axes[0].set_title("Windowed ROC AUC"); axes[0].set_xlabel("Instance"); axes[0].set_ylabel("AUC")
axes[1].set_title("Cumulative ROC AUC"); axes[1].set_xlabel("Instance"); axes[1].set_ylabel("AUC")
for ax in axes:
    ax.legend(fontsize=8)

plt.suptitle("Exp 2B – Effect of k (mixed-density, no drift)")
plt.tight_layout()
plt.show()

# Summary
rows = []
for k in k_values_2b:
    win = np.array(k_results_2b[k]["windowed"], dtype=float)
    valid = win[~np.isnan(win)]
    rows.append({
        "k": k,
        "Final cumulative AUC": k_results_2b[k]["cumulative"][-1],
        "Final windowed AUC": valid[-1] if len(valid) else np.nan,
        "Mean windowed AUC": valid.mean() if len(valid) else np.nan
    })
print(pd.DataFrame(rows).to_string(index=False))


**Interpretation:**
- **k = 80** achieves the highest and most stable windowed AUC — large neighborhoods allow LOF to see across density regions.
- Small k (3–10) is too local: neighborhoods are noisy and density contrasts are weak.
- The gap between k values is large and persistent, confirming **H4**: k is critical *when drift is not the dominant factor*.

**Synthesis with Exp 2A:**

| Setting | k matters? | Why? |
|---|---|---|
| Drift stream (2A) | Minimally | Drift dominates — all k converge to poor performance |
| Mixed-density static (2B) | Strongly | Local density structure drives detection — k controls granularity |

> LOF is sensitive to k, but the *importance* of that sensitivity is context-dependent.


---
### Exp 3 – LOF in Its Ideal Scenario: Local Anomalies

**Hypothesis tested:** H1  
**Goal:** Confirm that LOF *does* outperform HST when its assumptions hold — i.e., when anomalies are contextual (locally sparse) rather than globally extreme.

**Motivation:** Experiments 1 and 2 focused on limitations. For a balanced analysis, we must also show LOF in a setting designed for it. If LOF fails even here, the algorithm is fundamentally weak; if it succeeds, the project argument becomes much stronger: LOF is not universally bad — it degrades when its assumptions are violated.

**Dataset:** Two compact clusters, with anomalies placed in the low-density bridge between them. These points are not global outliers (they are near the centroid of the two clusters), but are locally unusual.


In [ ]:
def generate_local_anomaly_stream(n_samples=4000, seed=42):
    np.random.seed(seed)
    random.seed(seed)
    stream = []
    for _ in range(n_samples):
        r = random.random()
        if r < 0.90:
            center = random.choice([(0, 0), (4, 0)])
            x = np.random.normal(loc=center, scale=0.35, size=2); y = 0; phase = "normal"
        else:
            x = np.random.normal(loc=(2, 0), scale=0.18, size=2); y = 1; phase = "anomaly"
        stream.append(({"x1": float(x[0]), "x2": float(x[1])}, y, phase))
    return stream


stream_h1 = generate_local_anomaly_stream()
df_h1 = stream_to_df(stream_h1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df_h1[df_h1["label"]==0]["x1"], df_h1[df_h1["label"]==0]["x2"],
           s=8, alpha=0.3, label="Normal")
ax.scatter(df_h1[df_h1["label"]==1]["x1"], df_h1[df_h1["label"]==1]["x2"],
           s=20, alpha=0.8, c="red", label="Anomaly (bridge)")
ax.set_title("Exp 3 – Local-anomaly stream")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Use best k found in Exp 2B
lof_h1 = anomaly.LocalOutlierFactor(n_neighbors=80)
hst_h1 = anomaly.HalfSpaceTrees(n_trees=10, height=8, window_size=250)

cum_lof, win_lof = run_experiment_with_window(stream_h1, lof_h1, window_size=300)
cum_hst, win_hst = run_experiment_with_window(stream_h1, hst_h1, window_size=300)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(cum_lof, label="LOF (k=80)"); axes[0].plot(cum_hst, label="HST")
axes[0].set_title("Cumulative ROC AUC"); axes[0].set_xlabel("Instance"); axes[0].legend()

axes[1].plot(win_lof, label="LOF (k=80)"); axes[1].plot(win_hst, label="HST")
axes[1].set_title("Windowed ROC AUC"); axes[1].set_xlabel("Instance"); axes[1].legend()

plt.suptitle("Exp 3 – LOF vs HST on local-anomaly stream")
plt.tight_layout()
plt.show()

# Summary
win_lof_v = np.array(win_lof)[~np.isnan(win_lof)]
win_hst_v = np.array(win_hst)[~np.isnan(win_hst)]
results_h1 = pd.DataFrame({
    "Model": ["LOF (k=80)", "HST"],
    "Final cumulative AUC": [cum_lof[-1], cum_hst[-1]],
    "Final windowed AUC": [win_lof_v[-1], win_hst_v[-1]],
    "Mean windowed AUC": [win_lof_v.mean(), win_hst_v.mean()]
})
print(results_h1.to_string(index=False))


**Expected interpretation:**
- LOF with a well-tuned k should clearly outperform HST on this stream: the anomalies require local density comparison to distinguish.
- HST may assign low anomaly scores to bridge points because they are not globally isolated.
- This confirms **H1**: LOF is advantageous precisely when its core assumption — that anomalies are locally sparse — holds.


---
### Exp 4 – Real-World Validation: Credit Card Fraud

**Hypotheses tested:** H1, H2, H3 (external validity)  
**Goal:** Confirm whether the conclusions from synthetic experiments hold under realistic conditions (noise, overlap, true class imbalance).

**Plan:**
1. Run prequential evaluation of LOF (best k from Exp 2B) and HST on the full credit card stream.
2. Plot cumulative and windowed ROC AUC.
3. Compare final AUC values.
4. Discuss why performance may differ from synthetic experiments (no ground-truth phases, subtle anomalies, no clear local structure).

```python
# TODO: implement
# lof_cc = anomaly.LocalOutlierFactor(n_neighbors=80)
# hst_cc = anomaly.HalfSpaceTrees(n_trees=10, height=8, window_size=250)
# cum_lof_cc, win_lof_cc = run_experiment_with_window(creditcard_stream, lof_cc, window_size=500)
# cum_hst_cc, win_hst_cc = run_experiment_with_window(creditcard_stream, hst_cc, window_size=500)
```

> ⚠️ *Note:* The full credit card dataset (284K rows) may be slow with incremental LOF. Consider sub-sampling or capping at 50K–100K instances for an initial run, then discussing scalability implications.


---
### Exp 5 – Runtime & Memory

**Goal:** Assess practical streaming feasibility.

**Plan:**
1. Measure per-instance processing time for LOF (several k values) and HST.
2. Track memory consumption using `tracemalloc` or `memory_profiler`.
3. Plot: time per instance over the stream (to detect growth, if any).
4. Discuss complexity: LOF's neighborhood updates are O(n·k) in the worst case — relevant for long streams.

```python
# TODO: implement
# import time, tracemalloc
# for k in [10, 40, 80]:
#     lof = anomaly.LocalOutlierFactor(n_neighbors=k)
#     ... time each score_one / learn_one call ...
```

> ⚠️ *Note:* This experiment is critical for the streaming feasibility argument. An algorithm with excellent AUC but O(n²) memory growth is not usable in practice.


---
## 9. Discussion & Conclusions

*(To be completed after all experiments are run.)*

### Summary of hypothesis outcomes

| Hypothesis | Expected outcome | Evidence so far |
|---|---|---|
| H1 – LOF better for local anomalies | LOF > HST on local-anomaly stream | Partially confirmed (Exp 3) |
| H2 – LOF struggles under drift | LOF degrades at phase transitions | Confirmed (Exp 1) |
| H3 – HST more stable under drift | HST improves or holds steady | Confirmed (Exp 1) |
| H4 – LOF sensitive to k | Large k ≠ small k, especially in dense structures | Confirmed (Exp 2A+2B) |

### Key takeaways

- LOF is not universally weak — it works well when local density structure is meaningful and the stream is stationary.
- Drift is the primary limiting factor for LOF, not parameter sensitivity.
- k matters most when the stream has heterogeneous density (mixed clusters) and least when drift dominates.
- HST's advantage is consistency: it degrades less under distribution change.

### Limitations and future work

- LOF has no drift-handling mechanism. A **windowed or fading LOF** variant would be a natural extension to test.
- Runtime experiments (Exp 5) are needed to assess whether LOF's O(n·k) update cost is acceptable for high-throughput streams.
- Real-world validation (Exp 4) is needed to check whether synthetic findings generalise.
